# SOM World Map: จัดกลุ่มประเทศจากตัวชี้วัดโลก

Notebook นี้ทำโปรเจกต์ Self-Organizing Map (SOM) แบบครบจบในไฟล์เดียว โดยใช้ข้อมูลประเทศจาก World Bank แล้วแปลงข้อมูลหลายมิติให้กลายเป็นแผนที่ 2 มิติที่อ่านง่าย

**เป้าหมายของโปรเจกต์**

- ดึงข้อมูลประเทศจาก World Bank API
- แปลงข้อมูลหลายมิติให้เหมาะกับ SOM
- สร้าง SOM แบบไม่ต้องใช้ library สำเร็จรูป
- ดูว่าประเทศไหนมี profile คล้ายกัน เช่น ประเทศใกล้ไทยอยู่ตรงไหน
- อธิบายประโยชน์ของ SOM ผ่าน visualization

**แนวคิดสั้นๆ**

ข้อมูลประเทศหนึ่งประเทศมีหลายมิติ เช่น GDP, อายุคาดเฉลี่ย, Internet usage, urbanization, unemployment, electricity access และ CO2 ต่อหัว ถ้าดูทีละกราฟจะเห็นภาพรวมยากมาก SOM ช่วยจัดประเทศที่มี pattern คล้ายกันให้อยู่ใกล้กันบน grid 2 มิติ

## Outline

1. ตั้งค่าและดึงข้อมูลจาก World Bank
2. แปลงข้อมูลจากหลายปีและหลาย indicator ให้เป็นตารางประเทศ x feature
3. ทำ feature engineering และ scaling
4. train SOM จาก NumPy
5. visualize U-Matrix, component planes, macro clusters
6. วิเคราะห์ประเทศที่คล้ายไทยและสรุป insight

In [ ]:
from __future__ import annotations

import json
import math
import time
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

SEED = 42
rng = np.random.default_rng(SEED)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

ARTIFACT_DIR = PROJECT_ROOT
DATA_DIR = PROJECT_ROOT / "data"
RAW_JSON_DIR = DATA_DIR / "raw_world_bank_json"
FIG_DIR = PROJECT_ROOT / "figures"

for directory in [DATA_DIR, RAW_JSON_DIR, FIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 180,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

ARTIFACT_DIR

## 1. เลือกข้อมูลหลายมิติ

เราจะใช้ตัวชี้วัดที่ตีความง่ายและพออธิบาย profile ของประเทศได้:

- `NY.GDP.PCAP.CD`: GDP per capita
- `SP.DYN.LE00.IN`: Life expectancy
- `IT.NET.USER.ZS`: Internet users (% population)
- `SP.URB.TOTL.IN.ZS`: Urban population (% total)
- `SL.UEM.TOTL.ZS`: Unemployment (% labor force)
- `EG.ELC.ACCS.ZS`: Access to electricity (% population)
- `CC.CO2.EMSE.EL` + `SP.POP.TOTL`: CO2 total ปี 2018 หารด้วย population ปี 2018 เพื่อได้ CO2 tons per capita

เหตุผลที่ CO2 ทำแบบนี้: endpoint เก่า `EN.ATM.CO2E.PC` ถูก World Bank API ปัจจุบันตอบว่า archived/deleted แล้ว จึงใช้ชุด CCDR ที่ยังเรียกได้ และคำนวณต่อหัวเอง

In [ ]:
WORLD_BANK_BASE = "https://api.worldbank.org/v2"
DATE_RANGE = "2018:2024"

INDICATORS = {
    "gdp_per_capita": {
        "code": "NY.GDP.PCAP.CD",
        "label": "GDP per capita (current US$)",
        "how": "latest available in 2018-2024",
    },
    "life_expectancy": {
        "code": "SP.DYN.LE00.IN",
        "label": "Life expectancy at birth (years)",
        "how": "latest available in 2018-2024",
    },
    "internet_users_pct": {
        "code": "IT.NET.USER.ZS",
        "label": "Individuals using Internet (% of population)",
        "how": "latest available in 2018-2024",
    },
    "urban_population_pct": {
        "code": "SP.URB.TOTL.IN.ZS",
        "label": "Urban population (% of total)",
        "how": "latest available in 2018-2024",
    },
    "unemployment_pct": {
        "code": "SL.UEM.TOTL.ZS",
        "label": "Unemployment (% of labor force)",
        "how": "latest available in 2018-2024",
    },
    "electricity_access_pct": {
        "code": "EG.ELC.ACCS.ZS",
        "label": "Access to electricity (% of population)",
        "how": "latest available in 2018-2024",
    },
}

CO2_TOTAL_CODE = "CC.CO2.EMSE.EL"
POPULATION_CODE = "SP.POP.TOTL"
CO2_FEATURE = "co2_tons_per_capita_2018"

FEATURE_COLS = list(INDICATORS) + [CO2_FEATURE]

pd.DataFrame([
    {"feature": feature, **meta}
    for feature, meta in INDICATORS.items()
])

## 2. ดึงข้อมูลจาก World Bank API พร้อม cache

Cell ด้านล่างจะ cache raw JSON ลงใน `output/jupyter-notebook/som-world-map/data/raw_world_bank_json/` เพื่อให้รันซ้ำได้เร็วและยังเปิด notebook ได้แม้อินเทอร์เน็ตไม่เสถียร

In [ ]:
def cache_name(text: str) -> str:
    safe = "".join(ch if ch.isalnum() else "_" for ch in text)
    return f"{safe}.json"


def request_world_bank_json(url: str, cache_key: str, force_refresh: bool = False):
    cache_path = RAW_JSON_DIR / cache_name(cache_key)
    if cache_path.exists() and not force_refresh:
        return json.loads(cache_path.read_text(encoding="utf-8"))

    last_error = None
    for attempt in range(1, 5):
        try:
            response = requests.get(url, timeout=(10, 90))
            response.raise_for_status()
            data = response.json()
            cache_path.write_text(
                json.dumps(data, ensure_ascii=False, indent=2),
                encoding="utf-8",
            )
            return data
        except Exception as exc:
            last_error = exc
            wait_s = 1.5 * attempt
            print(f"Retry {attempt}/4 after {type(exc).__name__}: {wait_s:.1f}s")
            time.sleep(wait_s)

    raise RuntimeError(f"World Bank request failed: {url}") from last_error


def normalize_country_code(row: pd.Series) -> str:
    iso3 = row.get("countryiso3code", "")
    if isinstance(iso3, str) and iso3:
        return iso3
    country = row.get("country", {})
    return country.get("id", "")


def fetch_country_metadata() -> pd.DataFrame:
    url = f"{WORLD_BANK_BASE}/country?format=json&per_page=400"
    data = request_world_bank_json(url, "countries")
    countries = data[1]
    rows = []
    for item in countries:
        if item["region"]["value"] == "Aggregates":
            continue
        rows.append({
            "country_code": item["id"],
            "country_name": item["name"],
            "region": item["region"]["value"].strip(),
            "income_level": item["incomeLevel"]["value"],
            "capital_city": item["capitalCity"],
            "latitude": pd.to_numeric(item["latitude"], errors="coerce"),
            "longitude": pd.to_numeric(item["longitude"], errors="coerce"),
        })
    return pd.DataFrame(rows)


def fetch_indicator_long(code: str, date_range: str = DATE_RANGE) -> pd.DataFrame:
    url = (
        f"{WORLD_BANK_BASE}/country/all/indicator/{code}"
        f"?format=json&per_page=20000&date={date_range}"
    )
    data = request_world_bank_json(url, f"{code}_{date_range}")
    if isinstance(data, list) and data and isinstance(data[0], dict) and "message" in data[0]:
        message = data[0]["message"][0]["value"]
        raise ValueError(f"World Bank API error for {code}: {message}")

    df = pd.DataFrame(data[1])
    if df.empty:
        return df

    df["country_code"] = df.apply(normalize_country_code, axis=1)
    df["country_name"] = df["country"].apply(lambda x: x["value"])
    df["year"] = df["date"].astype(int)
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df[["country_code", "country_name", "year", "value"]]


country_meta = fetch_country_metadata()
country_meta.head()

## 3. แปลง API long format เป็นตารางประเทศ x feature

World Bank API คืนข้อมูลเป็น long format: 1 row = ประเทศ + ปี + indicator หนึ่งตัว  
สิ่งที่ต้องทำก่อนเข้า SOM:

1. ตัด aggregate เช่น World, East Asia & Pacific ออก เหลือเฉพาะประเทศจริง
2. สำหรับ indicator ทั่วไป ใช้ค่าล่าสุดที่ไม่เป็น missing ในช่วง 2018-2024
3. สำหรับ CO2 ใช้ total CO2 ปี 2018 จาก CCDR แล้วหาร population ปี 2018 เพื่อให้เทียบประเทศเล็ก/ใหญ่ได้ยุติธรรมขึ้น
4. เก็บเฉพาะประเทศที่มีข้อมูลครบทุก feature

In [ ]:
def latest_values_for_indicator(feature_name: str, code: str, valid_codes: set[str]) -> pd.DataFrame:
    df = fetch_indicator_long(code, DATE_RANGE)
    df = df[df["country_code"].isin(valid_codes) & df["value"].notna()].copy()
    latest = df.sort_values("year").groupby("country_code", as_index=False).tail(1)
    return latest[["country_code", "value", "year"]].rename(
        columns={"value": feature_name, "year": f"{feature_name}_year"}
    )


def build_country_feature_table() -> tuple[pd.DataFrame, pd.DataFrame]:
    valid_codes = set(country_meta["country_code"])
    full = country_meta.copy()

    for feature_name, meta in INDICATORS.items():
        latest = latest_values_for_indicator(feature_name, meta["code"], valid_codes)
        full = full.merge(latest, on="country_code", how="left")

    co2 = fetch_indicator_long(CO2_TOTAL_CODE, "2018:2018")
    pop = fetch_indicator_long(POPULATION_CODE, "2018:2018")

    co2 = co2[co2["country_code"].isin(valid_codes) & co2["value"].notna()].copy()
    pop = pop[pop["country_code"].isin(valid_codes) & pop["value"].notna()].copy()

    co2 = co2.rename(columns={"value": "co2_mt_total_2018"})
    pop = pop.rename(columns={"value": "population_2018"})

    co2_pc = co2[["country_code", "co2_mt_total_2018"]].merge(
        pop[["country_code", "population_2018"]],
        on="country_code",
        how="inner",
    )
    co2_pc[CO2_FEATURE] = (
        co2_pc["co2_mt_total_2018"] * 1_000_000 / co2_pc["population_2018"]
    )

    full = full.merge(
        co2_pc[["country_code", "co2_mt_total_2018", "population_2018", CO2_FEATURE]],
        on="country_code",
        how="left",
    )

    complete = full.dropna(subset=FEATURE_COLS).copy()
    return full, complete


full_table, model_table = build_country_feature_table()

full_table.to_csv(DATA_DIR / "world_bank_country_indicators_full.csv", index=False)
model_table.to_csv(DATA_DIR / "world_bank_som_input_complete_cases.csv", index=False)

print(f"ประเทศจริงทั้งหมดจาก metadata: {len(country_meta)}")
print(f"ประเทศที่มีข้อมูลครบทุก feature: {len(model_table)}")

missing_summary = (
    full_table[FEATURE_COLS]
    .isna()
    .sum()
    .rename("missing_countries")
    .to_frame()
)
display(missing_summary)

preview_cols = [
    "country_code", "country_name", "region", "income_level",
    *FEATURE_COLS,
]
display(model_table[preview_cols].head(10))

## 4. Feature engineering: ทำไมต้องแปลงข้อมูลก่อนเข้า SOM?

SOM ใช้ระยะทางระหว่าง vector ดังนั้น scale สำคัญมาก ถ้าเอา GDP ที่เป็นหลักหมื่นเข้าไปพร้อม unemployment ที่เป็นหลักหน่วยโดยไม่ scale ผลจะโดน GDP ครอบงำ

สิ่งที่ทำ:

- `log10(GDP per capita)` เพราะ GDP กระจายกว้างมาก
- `log1p(CO2 tons per capita)` เพราะ CO2 มี outlier สูง
- percentage features เก็บเป็นเปอร์เซ็นต์เดิมก่อน แล้วค่อย `StandardScaler`
- `StandardScaler` ทำให้ทุก feature มี mean 0 และ standard deviation 1

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    engineered = pd.DataFrame(index=df.index)
    engineered["log_gdp_per_capita"] = np.log10(df["gdp_per_capita"].clip(lower=1))
    engineered["life_expectancy"] = df["life_expectancy"]
    engineered["internet_users_pct"] = df["internet_users_pct"]
    engineered["urban_population_pct"] = df["urban_population_pct"]
    engineered["unemployment_pct"] = df["unemployment_pct"]
    engineered["electricity_access_pct"] = df["electricity_access_pct"]
    engineered["log1p_co2_tons_per_capita_2018"] = np.log1p(
        df[CO2_FEATURE].clip(lower=0)
    )
    return engineered


TRANSFORMED_FEATURE_COLS = [
    "log_gdp_per_capita",
    "life_expectancy",
    "internet_users_pct",
    "urban_population_pct",
    "unemployment_pct",
    "electricity_access_pct",
    "log1p_co2_tons_per_capita_2018",
]

transformed = engineer_features(model_table)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(transformed[TRANSFORMED_FEATURE_COLS])

scaled_df = pd.DataFrame(
    X_scaled,
    columns=[f"z_{col}" for col in TRANSFORMED_FEATURE_COLS],
    index=model_table.index,
)
scaled_output = pd.concat(
    [model_table[["country_code", "country_name", "region", "income_level"]], transformed, scaled_df],
    axis=1,
)
scaled_output.to_csv(DATA_DIR / "world_bank_som_scaled_features.csv", index=False)

display(transformed.describe().T.round(2))
print("Scaled feature means:", np.round(X_scaled.mean(axis=0), 4))
print("Scaled feature stds: ", np.round(X_scaled.std(axis=0), 4))

กราฟด้านล่างช่วยเช็กคร่าวๆ ว่า feature กระจายประมาณไหนหลังแปลง log แล้ว

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 6))
axes = axes.ravel()

for ax, col in zip(axes, TRANSFORMED_FEATURE_COLS):
    ax.hist(transformed[col], bins=24, color="#2f6f73", alpha=0.85, edgecolor="white")
    ax.set_title(col.replace("_", " "), fontsize=9)
    ax.tick_params(labelsize=8)

axes[-1].axis("off")
fig.suptitle("Feature distributions after log transforms", fontsize=12)
fig.tight_layout()
feature_dist_path = FIG_DIR / "feature_distributions.png"
fig.savefig(feature_dist_path, bbox_inches="tight")
plt.show()

feature_dist_path

## 5. สร้าง SOM จาก NumPy

SOM มี neuron เป็น grid 2 มิติ แต่ neuron แต่ละตัวเก็บ weight vector ที่มีจำนวนมิติเท่ากับ feature ของเรา  
ตอน train จะทำซ้ำ:

1. สุ่มประเทศหนึ่งประเทศ
2. หา Best Matching Unit (BMU) คือ neuron ที่ weight ใกล้ประเทศนั้นที่สุด
3. อัปเดต BMU และ neuron รอบๆ ให้ขยับเข้าใกล้ประเทศนั้น
4. ค่อยๆ ลด learning rate และ radius

In [ ]:
class SimpleSOM:
    def __init__(
        self,
        rows: int,
        cols: int,
        input_dim: int,
        learning_rate: float = 0.5,
        sigma: float | None = None,
        random_state: int = 42,
    ):
        self.rows = rows
        self.cols = cols
        self.input_dim = input_dim
        self.initial_learning_rate = learning_rate
        self.initial_sigma = sigma if sigma is not None else max(rows, cols) / 2
        self.rng = np.random.default_rng(random_state)
        self.weights = self.rng.normal(0, 1, size=(rows, cols, input_dim))
        rr, cc = np.indices((rows, cols))
        self.grid = np.stack([rr, cc], axis=-1)

    def initialize_from_data(self, data: np.ndarray) -> None:
        picks = self.rng.choice(data.shape[0], size=self.rows * self.cols, replace=True)
        weights = data[picks].reshape(self.rows, self.cols, self.input_dim)
        noise = self.rng.normal(0, 0.01, size=weights.shape)
        self.weights = weights + noise

    def find_bmu(self, x: np.ndarray) -> tuple[int, int]:
        distances = np.linalg.norm(self.weights - x, axis=2)
        return np.unravel_index(np.argmin(distances), (self.rows, self.cols))

    def train(self, data: np.ndarray, num_iterations: int = 8000) -> None:
        self.initialize_from_data(data)
        time_constant = num_iterations / math.log(self.initial_sigma + 1)

        for step in range(num_iterations):
            x = data[self.rng.integers(0, data.shape[0])]
            bmu = self.find_bmu(x)

            learning_rate = self.initial_learning_rate * math.exp(-step / num_iterations)
            sigma = self.initial_sigma * math.exp(-step / time_constant)
            sigma = max(sigma, 1e-4)

            grid_distance_sq = (
                (self.grid[..., 0] - bmu[0]) ** 2
                + (self.grid[..., 1] - bmu[1]) ** 2
            )
            influence = np.exp(-grid_distance_sq / (2 * sigma**2))[..., np.newaxis]
            self.weights += learning_rate * influence * (x - self.weights)

    def map_vectors(self, data: np.ndarray) -> np.ndarray:
        return np.array([self.find_bmu(x) for x in data])

    def quantization_error(self, data: np.ndarray) -> float:
        errors = []
        for x in data:
            bmu = self.find_bmu(x)
            errors.append(np.linalg.norm(x - self.weights[bmu]))
        return float(np.mean(errors))

    def topographic_error(self, data: np.ndarray) -> float:
        errors = 0
        flat_weights = self.weights.reshape(-1, self.input_dim)
        for x in data:
            distances = np.linalg.norm(flat_weights - x, axis=1)
            first, second = np.argsort(distances)[:2]
            r1, c1 = divmod(first, self.cols)
            r2, c2 = divmod(second, self.cols)
            if max(abs(r1 - r2), abs(c1 - c2)) > 1:
                errors += 1
        return errors / len(data)

    def u_matrix(self) -> np.ndarray:
        umat = np.zeros((self.rows, self.cols))
        for r in range(self.rows):
            for c in range(self.cols):
                neighbor_distances = []
                for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < self.rows and 0 <= nc < self.cols:
                        neighbor_distances.append(
                            np.linalg.norm(self.weights[r, c] - self.weights[nr, nc])
                        )
                umat[r, c] = np.mean(neighbor_distances)
        return umat

## 6. Train SOM

จำนวนประเทศที่ใช้ได้มีประมาณหลักร้อย ดังนั้น grid `9 x 10` พอให้เห็น topology โดยไม่ละเอียดเกินไป

In [ ]:
som = SimpleSOM(
    rows=9,
    cols=10,
    input_dim=X_scaled.shape[1],
    learning_rate=0.45,
    sigma=4.5,
    random_state=SEED,
)
som.train(X_scaled, num_iterations=9000)

bmu_coords = som.map_vectors(X_scaled)

results = model_table.copy().reset_index(drop=True)
results["som_row"] = bmu_coords[:, 0]
results["som_col"] = bmu_coords[:, 1]

qe = som.quantization_error(X_scaled)
te = som.topographic_error(X_scaled)

print(f"Quantization error: {qe:.3f}")
print(f"Topographic error:  {te:.3f}")
print(f"Countries mapped:   {len(results)}")

results[["country_code", "country_name", "som_row", "som_col"]].head()

**อ่าน metric ยังไง**

- Quantization error ต่ำแปลว่า neuron ตัวแทนประเทศได้ดีขึ้น
- Topographic error ต่ำแปลว่าโครงสร้างเพื่อนบ้านบน grid ยังรักษาความใกล้เคียงของข้อมูลเดิมได้ดี
- ค่า metric ไม่ใช่คะแนนสอบตายตัว จุดสำคัญของ SOM คือ visualization และการสำรวจ pattern

## 7. U-Matrix: แผนที่ระยะห่างระหว่าง neuron

U-Matrix แสดงว่า neuron ข้างๆ กันต่างกันมากแค่ไหน:

- สีเข้ม/สว่างสูง = รอยต่อระหว่างกลุ่ม
- สีต่ำ = พื้นที่ต่อเนื่องกัน ประเทศในบริเวณนั้นมักมี profile คล้ายกัน

In [ ]:
INCOME_ORDER = [
    "High income",
    "Upper middle income",
    "Lower middle income",
    "Low income",
    "Not classified",
]
INCOME_COLORS = {
    "High income": "#2a6fbb",
    "Upper middle income": "#1b9e77",
    "Lower middle income": "#f0a202",
    "Low income": "#d95f02",
    "Not classified": "#7f7f7f",
}

FOCUS_COUNTRIES = [
    "THA", "MYS", "VNM", "IDN", "PHL", "CHN", "JPN", "KOR",
    "IND", "USA", "DEU", "BRA", "MEX", "ZAF", "QAT", "SGP",
]


def jittered_positions(df: pd.DataFrame, jitter: float = 0.22) -> tuple[np.ndarray, np.ndarray]:
    local_rng = np.random.default_rng(SEED)
    x = df["som_col"].to_numpy(dtype=float) + local_rng.uniform(-jitter, jitter, len(df))
    y = df["som_row"].to_numpy(dtype=float) + local_rng.uniform(-jitter, jitter, len(df))
    return x, y


def plot_u_matrix(som: SimpleSOM, df: pd.DataFrame, focus_codes: list[str]) -> Path:
    umat = som.u_matrix()
    fig, ax = plt.subplots(figsize=(11, 8))
    image = ax.imshow(umat, cmap="magma_r", origin="upper")
    fig.colorbar(image, ax=ax, shrink=0.75, label="Average neighbor distance")

    x, y = jittered_positions(df)
    colors = df["income_level"].map(INCOME_COLORS).fillna("#7f7f7f")
    ax.scatter(x, y, s=36, c=colors, edgecolor="white", linewidth=0.55, alpha=0.92)

    focus = df[df["country_code"].isin(focus_codes)]
    for _, row in focus.iterrows():
        ax.text(
            row["som_col"],
            row["som_row"],
            row["country_code"],
            fontsize=8,
            weight="bold",
            ha="center",
            va="center",
            color="white",
            bbox=dict(boxstyle="round,pad=0.18", facecolor="black", alpha=0.72, linewidth=0),
        )

    patches = [
        mpatches.Patch(color=INCOME_COLORS[level], label=level)
        for level in INCOME_ORDER
        if level in set(df["income_level"])
    ]
    ax.legend(handles=patches, loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=False)
    ax.set_title("SOM U-Matrix: countries with similar profiles land near each other")
    ax.set_xlabel("SOM column")
    ax.set_ylabel("SOM row")
    ax.set_xticks(range(som.cols))
    ax.set_yticks(range(som.rows))
    ax.grid(color="white", alpha=0.18, linewidth=0.8)
    fig.tight_layout()

    path = FIG_DIR / "som_u_matrix_country_map.png"
    fig.savefig(path, bbox_inches="tight")
    plt.show()
    return path


u_matrix_path = plot_u_matrix(som, results, FOCUS_COUNTRIES)
u_matrix_path

## 8. Component planes: แต่ละบริเวณของแผนที่มี feature สูง/ต่ำอย่างไร

Component plane คือการดู weight ของแต่ละ feature บน SOM grid  
ตรงนี้ช่วยตีความว่าแถบไหนของแผนที่คือประเทศรายได้สูง, อายุยืน, internet สูง, CO2 สูง ฯลฯ

In [ ]:
FEATURE_LABELS = {
    "log_gdp_per_capita": "log GDP pc",
    "life_expectancy": "life expectancy",
    "internet_users_pct": "internet users",
    "urban_population_pct": "urban population",
    "unemployment_pct": "unemployment",
    "electricity_access_pct": "electricity access",
    "log1p_co2_tons_per_capita_2018": "log CO2 pc",
}


def plot_component_planes(som: SimpleSOM, feature_cols: list[str]) -> Path:
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    axes = axes.ravel()

    for idx, feature in enumerate(feature_cols):
        ax = axes[idx]
        plane = som.weights[:, :, idx]
        image = ax.imshow(plane, cmap="viridis", origin="upper")
        ax.set_title(FEATURE_LABELS.get(feature, feature), fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

    axes[-1].axis("off")
    fig.suptitle("SOM component planes, values are standardized feature weights", fontsize=12)
    fig.tight_layout()

    path = FIG_DIR / "som_component_planes.png"
    fig.savefig(path, bbox_inches="tight")
    plt.show()
    return path


component_path = plot_component_planes(som, TRANSFORMED_FEATURE_COLS)
component_path

## 9. Macro clusters จาก SOM grid

SOM เองจัดข้อมูลเป็นแผนที่ต่อเนื่องอยู่แล้ว แต่เพื่อเล่าในรายงานง่ายขึ้น เราสามารถ cluster neuron weights เป็นกลุ่มใหญ่ๆ ได้

In [ ]:
N_MACRO_CLUSTERS = 5
kmeans = KMeans(n_clusters=N_MACRO_CLUSTERS, random_state=SEED, n_init=20)
node_clusters = kmeans.fit_predict(som.weights.reshape(-1, som.input_dim)).reshape(som.rows, som.cols)

results["macro_cluster"] = [
    int(node_clusters[row, col])
    for row, col in results[["som_row", "som_col"]].to_numpy()
]

cluster_summary = (
    results.groupby("macro_cluster")
    .agg(
        countries=("country_code", "count"),
        avg_gdp_per_capita=("gdp_per_capita", "mean"),
        avg_life_expectancy=("life_expectancy", "mean"),
        avg_internet_users_pct=("internet_users_pct", "mean"),
        avg_urban_population_pct=("urban_population_pct", "mean"),
        avg_unemployment_pct=("unemployment_pct", "mean"),
        avg_electricity_access_pct=("electricity_access_pct", "mean"),
        avg_co2_tons_per_capita_2018=(CO2_FEATURE, "mean"),
    )
    .round(2)
    .sort_index()
)

examples = (
    results.sort_values(["macro_cluster", "country_name"])
    .groupby("macro_cluster")
    .head(8)
    .groupby("macro_cluster")["country_code"]
    .apply(lambda codes: ", ".join(codes))
    .rename("example_country_codes")
)

cluster_summary = cluster_summary.join(examples)
display(cluster_summary)

In [ ]:
def plot_macro_clusters(som: SimpleSOM, df: pd.DataFrame, node_clusters: np.ndarray) -> Path:
    cluster_colors = ["#3366aa", "#dd4477", "#66aa44", "#ffbb33", "#8e63ce"]
    cmap = plt.matplotlib.colors.ListedColormap(cluster_colors[:N_MACRO_CLUSTERS])

    fig, ax = plt.subplots(figsize=(11, 8))
    ax.imshow(node_clusters, cmap=cmap, origin="upper", alpha=0.42)

    x, y = jittered_positions(df)
    ax.scatter(
        x,
        y,
        s=34,
        c=df["macro_cluster"].map(lambda c: cluster_colors[int(c)]),
        edgecolor="white",
        linewidth=0.55,
        alpha=0.95,
    )

    for code in FOCUS_COUNTRIES:
        focus = df[df["country_code"] == code]
        if focus.empty:
            continue
        row = focus.iloc[0]
        ax.text(
            row["som_col"],
            row["som_row"],
            code,
            fontsize=8,
            weight="bold",
            ha="center",
            va="center",
            color="white",
            bbox=dict(boxstyle="round,pad=0.18", facecolor="black", alpha=0.72, linewidth=0),
        )

    handles = [
        mpatches.Patch(color=cluster_colors[i], label=f"Macro cluster {i}")
        for i in range(N_MACRO_CLUSTERS)
    ]
    ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=False)
    ax.set_title("Macro clusters on top of the SOM grid")
    ax.set_xlabel("SOM column")
    ax.set_ylabel("SOM row")
    ax.set_xticks(range(som.cols))
    ax.set_yticks(range(som.rows))
    ax.grid(color="white", alpha=0.25, linewidth=0.8)
    fig.tight_layout()

    path = FIG_DIR / "som_macro_clusters.png"
    fig.savefig(path, bbox_inches="tight")
    plt.show()
    return path


macro_path = plot_macro_clusters(som, results, node_clusters)
macro_path

## 10. ประเทศที่คล้ายไทย

เราดูได้ 2 แบบ:

- `feature_distance`: ระยะในข้อมูลหลายมิติหลัง scale แล้ว ยิ่งต่ำยิ่งคล้าย
- `som_grid_distance`: ระยะบน SOM grid ยิ่งต่ำยิ่งอยู่โซนเดียวกัน

In [ ]:
def similar_countries(target_code: str, top_n: int = 12) -> pd.DataFrame:
    target_code = target_code.upper()
    if target_code not in set(results["country_code"]):
        raise ValueError(f"{target_code} is not in the complete-case dataset")

    target_idx = results.index[results["country_code"] == target_code][0]
    feature_distance = np.linalg.norm(X_scaled - X_scaled[target_idx], axis=1)
    target_row = results.loc[target_idx, "som_row"]
    target_col = results.loc[target_idx, "som_col"]
    grid_distance = (
        (results["som_row"] - target_row).abs()
        + (results["som_col"] - target_col).abs()
    )

    out = results.copy()
    out["feature_distance"] = feature_distance
    out["som_grid_distance"] = grid_distance
    cols = [
        "country_code", "country_name", "region", "income_level",
        "som_row", "som_col", "macro_cluster",
        "feature_distance", "som_grid_distance",
        "gdp_per_capita", "life_expectancy", "internet_users_pct",
        "urban_population_pct", "unemployment_pct",
        "electricity_access_pct", CO2_FEATURE,
    ]
    return out.sort_values("feature_distance")[cols].head(top_n)


thailand_neighbors = similar_countries("THA", top_n=12)
thailand_neighbors.to_csv(DATA_DIR / "thailand_similar_countries.csv", index=False)
display(thailand_neighbors.round({
    "feature_distance": 3,
    "gdp_per_capita": 0,
    "life_expectancy": 1,
    "internet_users_pct": 1,
    "urban_population_pct": 1,
    "unemployment_pct": 1,
    "electricity_access_pct": 1,
    CO2_FEATURE: 2,
}))

In [ ]:
def plot_target_neighbors(target_code: str, top_n: int = 12) -> Path:
    neighbors = similar_countries(target_code, top_n=top_n).copy()
    target_code = target_code.upper()
    distances = neighbors.sort_values("feature_distance", ascending=True)

    fig, ax = plt.subplots(figsize=(10, 5.5))
    colors = ["#d95f02" if code == target_code else "#2a6fbb" for code in distances["country_code"]]
    ax.barh(distances["country_code"], distances["feature_distance"], color=colors)
    ax.invert_yaxis()
    ax.set_xlabel("Distance in scaled multi-dimensional feature space")
    ax.set_title(f"Most similar countries to {target_code}")
    for i, (_, row) in enumerate(distances.iterrows()):
        label = f"{row['country_name']} | SOM ({row['som_row']}, {row['som_col']})"
        ax.text(row["feature_distance"] + 0.03, i, label, va="center", fontsize=8)
    fig.tight_layout()

    path = FIG_DIR / f"similar_countries_{target_code}.png"
    fig.savefig(path, bbox_inches="tight")
    plt.show()
    return path


thailand_neighbors_path = plot_target_neighbors("THA", top_n=12)
thailand_neighbors_path

## 11. สรุปผลลัพธ์เป็นไฟล์

Cell นี้รวมผลลัพธ์ที่สำคัญและบันทึก CSV ที่ใช้ส่งหรือเอาไปทำ slide ต่อได้

In [ ]:
output_cols = [
    "country_code", "country_name", "region", "income_level",
    "som_row", "som_col", "macro_cluster",
    *FEATURE_COLS,
]

results[output_cols].to_csv(DATA_DIR / "country_som_results.csv", index=False)
cluster_summary.to_csv(DATA_DIR / "macro_cluster_summary.csv")

artifact_index = pd.DataFrame({
    "artifact": [
        "full input table",
        "complete-case SOM input",
        "scaled features",
        "country SOM results",
        "macro cluster summary",
        "Thailand neighbors",
        "feature distributions",
        "U-Matrix country map",
        "component planes",
        "macro cluster map",
        "Thailand neighbor chart",
    ],
    "path": [
        DATA_DIR / "world_bank_country_indicators_full.csv",
        DATA_DIR / "world_bank_som_input_complete_cases.csv",
        DATA_DIR / "world_bank_som_scaled_features.csv",
        DATA_DIR / "country_som_results.csv",
        DATA_DIR / "macro_cluster_summary.csv",
        DATA_DIR / "thailand_similar_countries.csv",
        FIG_DIR / "feature_distributions.png",
        FIG_DIR / "som_u_matrix_country_map.png",
        FIG_DIR / "som_component_planes.png",
        FIG_DIR / "som_macro_clusters.png",
        FIG_DIR / "similar_countries_THA.png",
    ],
})

display(artifact_index)
print("Saved artifacts under:", ARTIFACT_DIR)

## 12. Insight สำหรับเขียนรายงาน

สิ่งที่โปรเจกต์นี้แสดงให้เห็น:

- SOM เปลี่ยนข้อมูลประเทศ 7 มิติให้เป็นแผนที่ 2 มิติที่ดูเพื่อนบ้านและกลุ่มได้ง่าย
- ประเทศที่อยู่ใกล้กันบน SOM ไม่ได้แปลว่าอยู่ใกล้กันทางภูมิศาสตร์ แต่แปลว่า profile ของตัวชี้วัดคล้ายกัน
- U-Matrix ช่วยเห็น boundary ระหว่างกลุ่มประเทศ
- Component planes ช่วยอธิบายว่าแต่ละโซนของแผนที่เด่นเรื่องอะไร เช่น GDP สูง, Internet สูง, CO2 สูง หรือ unemployment สูง
- การ normalize สำคัญมาก เพราะ SOM ใช้ระยะทาง ถ้าไม่ scale ตัวแปรที่เลขใหญ่จะ dominate ผลลัพธ์

**ข้อควรระวัง**

- ผลลัพธ์ขึ้นกับ feature ที่เลือก ถ้าเพิ่ม education หรือ health expenditure แผนที่จะเปลี่ยนได้
- ปีของข้อมูลไม่เท่ากันทุก indicator จึงใช้ latest available และระบุวิธีเลือกปีไว้
- SOM เหมาะกับ exploratory analysis มากกว่าการตัดสินว่า cluster ใด “ถูกต้องที่สุด”

## Exercise เล็กๆ

ลองเปลี่ยนประเทศเป้าหมายแล้วดูประเทศที่คล้ายกัน เช่น `JPN`, `USA`, `BRA`, `IND`, `MYS`

In [ ]:
# เปลี่ยน target_code ได้ เช่น "JPN", "USA", "BRA", "IND", "MYS"
target_code = "MYS"
display(similar_countries(target_code, top_n=10).round(3))

## แหล่งข้อมูล

- World Bank Indicators API: https://datahelpdesk.worldbank.org/knowledgebase/articles/889392
- World Development Indicators catalog: https://datacatalog.worldbank.org/infrastructure-data/search/dataset/0037712/World-Development-Indicators
- World Bank API endpoint pattern: `https://api.worldbank.org/v2/country/all/indicator/{indicator_code}?format=json`